In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('nhanes_data')
CYCLES = {'2011-12':'G', '2013-14':'H', '2015-16':'I', '2017-18':'J'}
YEAR_MAP = {'2011-12':'2011', '2013-14':'2013', '2015-16':'2015', '2017-18':'2017'}

def load_nhanes_file(filename_pattern, cols_map):
    """Load an XPT file using pandas and rename columns per cols_map."""
    frames = []
    for cycle, suffix in CYCLES.items():
        fname = filename_pattern.format(suffix)
        f = DATA_DIR / fname
        if not f.exists():
            print(f'  Missing: {fname} — skipping')
            continue
        try:
            df = pd.read_sas(str(f), format='xport', encoding='utf-8')
            keep = [c for c in cols_map if c in df.columns]
            df = df[keep].rename(columns={k:v for k,v in cols_map.items() if k in keep})
            frames.append(df)
        except Exception as e:
            print(f'  Could not read {fname}: {e}')
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

print('Setup complete.')

Setup complete.


In [4]:

import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('nhanes_data')
CYCLES = {'2011-12':'G', '2013-14':'H', '2015-16':'I', '2017-18':'J'}

DR_COLS = {
    'SEQN':    'participant_id',
    'WTDR2D':  'dietary_weight',
    'DR1TKCAL':'energy_day1_kcal',
    'DR1TCARB':'carb_day1_g',
    'DR1TFIBE':'fiber_day1_g',
    'DR1TTFAT':'fat_total_day1_g',
    'DR1TSFAT':'sat_fat_day1_g',
    'DR1TMFAT':'mufa_day1_g',
    'DR1TPFAT':'pufa_day1_g',
    'DR1TPROT':'protein_day1_g',
    'DR1TSODI':'sodium_day1_g',   # stored as _g for loop consistency
    'DR1TSUGR':'sugars_day1_g',
}

DR2_COLS = {
    'SEQN':    'participant_id',
    'WTDR2D':  'dietary_weight',
    'DR2TKCAL':'energy_day2_kcal',
    'DR2TCARB':'carb_day2_g',
    'DR2TFIBE':'fiber_day2_g',
    'DR2TTFAT':'fat_total_day2_g',
    'DR2TSFAT':'sat_fat_day2_g',
    'DR2TMFAT':'mufa_day2_g',
    'DR2TPFAT':'pufa_day2_g',
    'DR2TPROT':'protein_day2_g',
    'DR2TSODI':'sodium_day2_g',   # same fix here
    'DR2TSUGR':'sugars_day2_g',
}

NUTRIENT_COLS = [
    'energy', 'carb', 'fiber', 'fat_total', 'sat_fat',
    'mufa', 'pufa', 'protein', 'sodium', 'sugars'
]

frames = []
for cycle, suffix in CYCLES.items():
    f1 = DATA_DIR / f'DR1TOT_{suffix}.XPT'
    f2 = DATA_DIR / f'DR2TOT_{suffix}.XPT'
    if not f1.exists():
        print(f'Missing {f1.name} — skipping {cycle}')
        continue

    df1 = pd.read_sas(str(f1), format='xport', encoding='utf-8')
    df1 = df1[[c for c in DR_COLS if c in df1.columns]].rename(columns=DR_COLS)

    if f2.exists():
        df2 = pd.read_sas(str(f2), format='xport', encoding='utf-8')
        df2 = df2[[c for c in DR2_COLS if c in df2.columns]].rename(columns=DR2_COLS)
        merged = df1.merge(df2, on='participant_id', suffixes=('','_r'))

        for col in NUTRIENT_COLS:
            c1 = f'{col}_day1_g' if col != 'energy' else 'energy_day1_kcal'
            c2 = f'{col}_day2_g' if col != 'energy' else 'energy_day2_kcal'
            if c1 in merged.columns and c2 in merged.columns:
                merged[f'{col}_avg'] = merged[[c1, c2]].mean(axis=1)
            elif c1 in merged.columns:
                merged[f'{col}_avg'] = merged[c1]

        df_cycle = merged[['participant_id','dietary_weight'] +
                          [c for c in merged.columns if c.endswith('_avg')]]
    else:
        for col in NUTRIENT_COLS:
            c1 = f'{col}_day1_g' if col != 'energy' else 'energy_day1_kcal'
            if c1 in df1.columns:
                df1[f'{col}_avg'] = df1[c1]
        df_cycle = df1[['participant_id','dietary_weight'] +
                       [c for c in df1.columns if c.endswith('_avg')]]

    df_cycle['cycle'] = cycle
    frames.append(df_cycle)
    print(f'Cycle {cycle}: {len(df_cycle)} participants')

dietary = pd.concat(frames, ignore_index=True)
print(f'\nTotal dietary records: {len(dietary)}')
dietary.head()


Cycle 2011-12: 9338 participants
Cycle 2013-14: 9813 participants
Cycle 2015-16: 9544 participants
Cycle 2017-18: 8704 participants

Total dietary records: 37399


,participant_id,dietary_weight,energy_avg,carb_avg,fiber_avg,fat_total_avg,sat_fat_avg,mufa_avg,pufa_avg,protein_avg,sodium_avg,sugars_avg,cycle
0,62161.0,46955.912018,3268.0,411.180,19.90,135.940,32.9230,52.2185,40.5475,107.465,4329.0,170.730,2011-12
1,62162.0,13901.170266,1516.0,214.935,12.60,51.710,18.7840,17.8565,11.6300,50.685,2462.0,106.115,2011-12
2,62163.0,2755.061606,1783.0,232.485,9.15,62.290,21.9275,22.6900,12.1350,73.225,3643.0,88.840,2011-12
3,62164.0,52879.479478,1743.5,172.660,17.60,40.130,8.7270,15.4250,11.9715,79.460,3555.0,40.630,2011-12
4,62165.0,9478.001856,1406.5,166.770,8.75,56.115,13.5830,18.0765,20.4415,62.730,2046.0,69.860,2011-12


In [5]:
def load_nhanes_file(filename, cols_map):
    """Load an XPT file and rename columns per cols_map."""
    frames = []
    for suffix in CYCLES.values():
        f = DATA_DIR / filename.format(suffix)
        if not f.exists(): continue
        df, _ = pyreadstat.read_xport(str(f))
        keep = [c for c in cols_map if c in df.columns]
        df = df[keep].rename(columns={k:v for k,v in cols_map.items() if k in keep})
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

# Demographics
demo = load_nhanes_file('DEMO_{}.XPT', {
    'SEQN':   'participant_id',
    'RIAGENDR': 'gender',       # 1=male, 2=female
    'RIDAGEYR': 'age_years',
    'RIDRETH3': 'race_ethnicity', # 6=non-Hispanic Asian
    'INDFMPIR': 'poverty_ratio',
})

# HbA1c (diabetes outcome: >= 6.5% = diabetic, >= 5.7% = prediabetic)
hba1c = load_nhanes_file('GHB_{}.XPT', {
    'SEQN':   'participant_id',
    'LBXGH':  'hba1c_pct',
})

# Lipids (CVD outcome)
tchol = load_nhanes_file('TCHOL_{}.XPT', {'SEQN':'participant_id', 'LBXTC':'total_chol'})
hdl   = load_nhanes_file('HDL_{}.XPT',   {'SEQN':'participant_id', 'LBDHDD':'hdl'})
trigly = load_nhanes_file('TRIGLY_{}.XPT',{'SEQN':'participant_id', 'LBXTR':'triglycerides'})

# BMI
bmx = load_nhanes_file('BMX_{}.XPT', {
    'SEQN':   'participant_id',
    'BMXBMI': 'bmi',
    'BMXWAIST':'waist_cm',
})

print('Demo:', demo.shape, '| HbA1c:', hba1c.shape)
print('Lipids:', tchol.shape, '| BMX:', bmx.shape)


Demo: (39156, 5) | HbA1c: (26673, 2)
Lipids: (31568, 2) | BMX: (37399, 3)


In [6]:
# Merge all files on participant_id
df = dietary.copy()
for other in [demo, hba1c, tchol, hdl, trigly, bmx]:
    if len(other) > 0:
        df = df.merge(other, on='participant_id', how='left')

print(f'Merged dataset: {df.shape}')

# Create outcome variables
# Diabetes: HbA1c >= 5.7 (prediabetes threshold — more actionable than 6.5)
df['outcome_diabetes'] = (df['hba1c_pct'] >= 5.7).astype(int)

# CVD risk: non-HDL cholesterol > 130 mg/dL (LDL proxy)
# non-HDL = total_chol - HDL (more reliable than LDL directly in NHANES)
df['non_hdl'] = df['total_chol'] - df['hdl']
df['outcome_cvd'] = (df['non_hdl'] > 130).astype(int)

# Keep only adults 20+ with dietary data
df = df[df['age_years'] >= 20].copy()
df = df[df['energy_avg'].notna() & (df['energy_avg'] > 500)].copy()

print(f'After filtering (adults with dietary data): {df.shape}')
print(f'Diabetes outcome prevalence: {df["outcome_diabetes"].mean():.1%}')
print(f'CVD risk outcome prevalence: {df["outcome_cvd"].mean():.1%}')

# Asian subsample
asian = df[df['race_ethnicity'] == 6].copy()
print(f'\nNon-Hispanic Asian subsample: {len(asian)} participants')
print(f'Asian diabetes prevalence: {asian["outcome_diabetes"].mean():.1%}')

# Save cleaned dataset
df.to_csv('nhanes_data/nhanes_cleaned.csv', index=False)
print('\nSaved: nhanes_data/nhanes_cleaned.csv')


Merged dataset: (37399, 23)
After filtering (adults with dietary data): (19509, 26)
Diabetes outcome prevalence: 39.8%
CVD risk outcome prevalence: 50.2%

Non-Hispanic Asian subsample: 2278 participants
Asian diabetes prevalence: 38.2%

Saved: nhanes_data/nhanes_cleaned.csv
